In [1]:
from pathlib import Path
from langchain.llms import Ollama
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

# 1. Prepare list of images
img_dir = Path("output/images_cleaned")
img_paths = list(img_dir.rglob("*.[pjPj][pnN][gG]"))
print(f"🖼️ {len(img_paths)} images found in {img_dir}")

# 2. Initialize the LLM
llm = Ollama(model="llava")

# 3. Description function with retry
def generate_description(img_path, max_retries=3):
    for attempt in range(max_retries):
        try:
            prompt = f"Describe this image in English:\n![image]({img_path.resolve()})"
            result = llm.invoke(input=prompt)
            return {
                "source": str(img_path.relative_to("output")),
                "description": result.strip()
            }
        except Exception as e:
            time.sleep(1)
    return {
        "source": str(img_path.relative_to("output")),
        "description": "Error: Failed after retries"
    }

# 4. Run with ThreadPoolExecutor and tqdm progress bar
max_workers = 16
image_docs = []
start_time = time.time()

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(generate_description, img): img for img in img_paths}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Generating Descriptions", ncols=100):
        image_docs.append(future.result())

# 5. Save results
with open("output/image_descriptions.json", "w", encoding="utf-8") as f:
    json.dump(image_docs, f, indent=2, ensure_ascii=False)

elapsed = time.time() - start_time
print(f"\n✅ {len(image_docs)} descriptions generated in {elapsed/60:.2f} minutes.")
print("💾 Saved to output/image_descriptions.json")


C:\Users\user\AppData\Local\Temp\ipykernel_19004\3666786060.py:14: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llava")


🖼️ 1230 images found in output\images_cleaned


Generating Descriptions: 100%|███████████████████████████████| 1230/1230 [21:09:48<00:00, 61.94s/it]


✅ 1230 descriptions generated in 1269.80 minutes.
💾 Saved to output/image_descriptions.json


In [8]:
import json
from pathlib import Path

# Entrées
input_data_path = Path("output/data_cleaned.json")
input_desc_path = Path("output/image_descriptions.json")
output_path = Path("output/metadata_merged.json")

# Chargement des fichiers
with open(input_data_path, "r", encoding="utf-8") as f:
    data_cleaned = json.load(f)

with open(input_desc_path, "r", encoding="utf-8") as f:
    desc_data = json.load(f)

# Normaliser les chemins et créer un dictionnaire {image_path: description}
desc_dict = {
    Path(d["source"].replace("\\", "/")).as_posix(): d["description"]
    for d in desc_data
}

# Traitement et fusion
merged = []
for item in data_cleaned:
    # Supprimer le champ 'raster'
    item.pop("raster", None)

    # Normaliser les chemins d'images
    images = [Path(img.replace("\\", "/")).as_posix() for img in item.get("images", [])]

    # Associer chaque image à sa description si disponible
    image_info = []
    for img_path in images:
        image_info.append({
            "path": img_path,
            "description": desc_dict.get(img_path, "No description available")
        })

    # Ajouter les infos fusionnées
    item["image_info"] = image_info
    # Retirer l'ancien champ 'images'
    item.pop("images", None)

    merged.append(item)

# Sauvegarde
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=2, ensure_ascii=False)

print(f"✅ Fusion réussie : {len(merged)} pages enregistrées dans {output_path}")


✅ Fusion réussie : 7591 pages enregistrées dans output\metadata_merged.json


In [1]:
import json
import re
from typing import List, Dict
from langchain.text_splitter import RecursiveCharacterTextSplitter

def load_metadata(file_path: str) -> List[Dict]:
    """Charger le fichier JSON de métadonnées"""
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def intelligent_chunk_text(text: str, max_chunk_size: int = 500) -> List[str]:
    """
    Découpe intelligente du texte en chunks
    Préserve les sections logiques (titres, paragraphes)
    """
    # Splitter configuré pour préserver la structure
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chunk_size,
        chunk_overlap=50,  # Overlap pour maintenir le contexte
        length_function=len,
        separators=[
            "\n\n",      # Paragraphes
            "\n",        # Lignes
            ".",         # Phrases
            " ",         # Mots
            ""
        ]
    )
    
    return text_splitter.split_text(text)

def create_chunks_with_metadata(metadata_list: List[Dict]) -> List[Dict]:
    """
    Crée des chunks avec métadonnées préservées
    Chaque chunk garde l'info sur les images de la page
    """
    all_chunks = []
    chunk_id = 0
    
    for page_data in metadata_list:
        pdf_name = page_data['pdf']
        pdf_id = page_data['pdf_id']
        page_num = page_data['page']
        full_text = page_data['text']
        images = page_data.get('image_info', [])
        
        # Découper le texte en chunks
        text_chunks = intelligent_chunk_text(full_text)
        
        # Créer un chunk pour chaque segment
        for i, chunk_text in enumerate(text_chunks):
            chunk = {
                'chunk_id': f"{pdf_id}_p{page_num}_c{i}",
                'pdf': pdf_name,
                'pdf_id': pdf_id,
                'page': page_num,
                'chunk_index': i,
                'text': chunk_text.strip(),
                'images': images,  # Toutes les images de la page sont associées
                'metadata': {
                    'source': f"{pdf_name} - Page {page_num}",
                    'chunk_size': len(chunk_text),
                    'has_images': len(images) > 0,
                    'image_count': len(images)
                }
            }
            all_chunks.append(chunk)
            chunk_id += 1
    
    return all_chunks

def save_chunks(chunks: List[Dict], output_path: str):
    """Sauvegarder les chunks"""
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    
    print(f"✅ {len(chunks)} chunks sauvegardés dans {output_path}")

def analyze_chunks(chunks: List[Dict]):
    """Analyser les statistiques des chunks"""
    total_chunks = len(chunks)
    chunks_with_images = sum(1 for c in chunks if c['metadata']['has_images'])
    total_images = sum(c['metadata']['image_count'] for c in chunks)
    avg_chunk_size = sum(c['metadata']['chunk_size'] for c in chunks) / total_chunks
    
    print(f"\n📊 STATISTIQUES DES CHUNKS:")
    print(f"Total chunks: {total_chunks}")
    print(f"Chunks avec images: {chunks_with_images} ({chunks_with_images/total_chunks*100:.1f}%)")
    print(f"Total images: {total_images}")
    print(f"Taille moyenne chunk: {avg_chunk_size:.0f} caractères")

# UTILISATION
if __name__ == "__main__":
    # 1. Charger vos métadonnées
    metadata = load_metadata("output/metadata_merged.json")
    print(f"📖 Chargé {len(metadata)} pages")
    
    # 2. Créer les chunks
    chunks = create_chunks_with_metadata(metadata)
    
    # 3. Analyser
    analyze_chunks(chunks)
    
    # 4. Sauvegarder
    save_chunks(chunks, "output/chunks_prepared.json")
    
    # 5. Exemple de chunk
    print(f"\n🔍 EXEMPLE DE CHUNK:")
    print(json.dumps(chunks[0], ensure_ascii=False, indent=2))

📖 Chargé 7591 pages

📊 STATISTIQUES DES CHUNKS:
Total chunks: 31021
Chunks avec images: 7033 (22.7%)
Total images: 10033
Taille moyenne chunk: 413 caractères
✅ 31021 chunks sauvegardés dans output/chunks_prepared.json

🔍 EXEMPLE DE CHUNK:
{
  "chunk_id": "100_more_swimming_drills_p2_c0",
  "pdf": "100 MORE SWIMMING DRILLS.pdf",
  "pdf_id": "100_more_swimming_drills",
  "page": 2,
  "chunk_index": 0,
  "text": "About the book\n100\nMoRe\nSWIMMING\nDRILLS\nBlythe\nLucero\nsAMple exeRcise the AuthoR\nTIp FORWARD\nThe purpose of this drill\nTo swim better, we have to swim more efficiently. While good technique is the foundation Blythe Lucero has been coaching swimming for more\n• Achieving a downhill floating position of efficient swimming, it is difficult to achieve by simply swimming lap after lap. Ongoing than 25 years. She currently oversees two teams,",
  "images": [
    {
      "path": "images_cleaned/100_more_swimming_drills/100_more_swimming_drills_p002_img1.jpeg",
      "descripti

In [2]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import pickle
from typing import List, Dict
from tqdm import tqdm
import faiss
import re

class SwimmingOptimizedRAG:
    def __init__(self):
        """
        RAG optimisé spécifiquement pour contenu natation en anglais
        """
        print("🏊‍♂️ Chargement du modèle optimisé pour natation...")
        # Modèle excellent pour contenu technique anglais
        self.model = SentenceTransformer('all-mpnet-base-v2')
        self.chunks = []
        self.embeddings = None
        self.index = None
        
        # Vocabulaire spécialisé natation
        self.swimming_terms = {
            # Strokes/Nages
            'freestyle': ['freestyle', 'front crawl', 'crawl stroke'],
            'backstroke': ['backstroke', 'back crawl', 'back stroke'],
            'breaststroke': ['breaststroke', 'breast stroke', 'frog stroke'],
            'butterfly': ['butterfly', 'fly stroke', 'dolphin stroke'],
            
            # Techniques
            'technique': ['technique', 'form', 'mechanics', 'stroke technique'],
            'breathing': ['breathing', 'breath', 'bilateral breathing'],
            'kick': ['kick', 'flutter kick', 'dolphin kick', 'frog kick'],
            'pull': ['pull', 'arm stroke', 'catch', 'pull phase'],
            
            # Training/Entraînement
            'drill': ['drill', 'exercise', 'practice', 'training drill'],
            'workout': ['workout', 'training', 'practice', 'set'],
            'beginner': ['beginner', 'novice', 'learning', 'basic'],
            'advanced': ['advanced', 'competitive', 'elite', 'expert'],
            
            # Equipment/Équipement
            'equipment': ['kickboard', 'pull buoy', 'fins', 'paddles', 'goggles'],
            
            # Performance
            'speed': ['speed', 'fast', 'sprint', 'velocity'],
            'endurance': ['endurance', 'distance', 'aerobic', 'stamina']
        }
        
        print("✅ Modèle spécialisé natation chargé !")
    
    def load_swimming_chunks(self, file_path: str):
        """Charger tous les chunks de natation"""
        with open(file_path, 'r', encoding='utf-8') as f:
            self.chunks = json.load(f)
        
        print(f"🏊‍♂️ {len(self.chunks)} chunks de natation chargés")
        
        # Analyser le contenu natation
        self.analyze_swimming_content()
    
    def analyze_swimming_content(self):
        """Analyser le contenu spécifique à la natation"""
        categories = {
            'technique': 0, 'drills': 0, 'beginner': 0, 'advanced': 0,
            'freestyle': 0, 'backstroke': 0, 'breaststroke': 0, 'butterfly': 0,
            'with_images': 0
        }
        
        for chunk in self.chunks:
            text_lower = chunk['text'].lower()
            
            # Compter les catégories
            if any(term in text_lower for term in ['technique', 'form', 'mechanics']):
                categories['technique'] += 1
            if any(term in text_lower for term in ['drill', 'exercise', 'practice']):
                categories['drills'] += 1
            if any(term in text_lower for term in ['beginner', 'novice', 'learning']):
                categories['beginner'] += 1
            if any(term in text_lower for term in ['advanced', 'competitive', 'elite']):
                categories['advanced'] += 1
            
            # Compter les nages
            if any(term in text_lower for term in ['freestyle', 'front crawl', 'crawl']):
                categories['freestyle'] += 1
            if any(term in text_lower for term in ['backstroke', 'back']):
                categories['backstroke'] += 1
            if any(term in text_lower for term in ['breaststroke', 'breast']):
                categories['breaststroke'] += 1
            if any(term in text_lower for term in ['butterfly', 'fly']):
                categories['butterfly'] += 1
            
            if chunk['metadata']['has_images']:
                categories['with_images'] += 1
        
        print(f"\n🏊‍♂️ ANALYSE CONTENU NATATION:")
        print(f"📚 Techniques: {categories['technique']} chunks")
        print(f"🏃‍♂️ Exercices: {categories['drills']} chunks")
        print(f"👶 Débutant: {categories['beginner']} chunks")
        print(f"🏆 Avancé: {categories['advanced']} chunks")
        print(f"🏊‍♀️ Freestyle: {categories['freestyle']} chunks")
        print(f"🏊‍♂️ Backstroke: {categories['backstroke']} chunks")
        print(f"🐸 Breaststroke: {categories['breaststroke']} chunks")
        print(f"🦋 Butterfly: {categories['butterfly']} chunks")
        print(f"🖼️  Avec images: {categories['with_images']} chunks")
    
    def optimize_swimming_text(self, chunk: Dict) -> str:
        """
        Optimiser le texte spécifiquement pour la natation
        """
        text = chunk['text']
        
        # Nettoyer le texte
        text = re.sub(r'\n+', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        
        # Identifier le type de contenu natation
        swimming_type = self.identify_swimming_type(text)
        stroke_type = self.identify_stroke(text)
        level = self.identify_level(text)
        
        # Métadonnées enrichies
        pdf_name = chunk['pdf'].replace('.pdf', '').replace('_', ' ')
        
        # Texte optimisé pour recherche natation
        optimized = f"[{swimming_type}] [{stroke_type}] [{level}] {pdf_name} Page {chunk['page']}: {text}"
        
        # Ajouter info images avec contexte natation
        if chunk['metadata']['has_images']:
            optimized += f" [Visual demonstration with {chunk['metadata']['image_count']} swimming illustrations]"
        
        return optimized
    
    def identify_swimming_type(self, text: str) -> str:
        """Identifier le type de contenu natation"""
        text_lower = text.lower()
        
        if any(term in text_lower for term in ['drill', 'exercise', 'practice']):
            return "Swimming-Drill"
        elif any(term in text_lower for term in ['technique', 'form', 'mechanics', 'stroke']):
            return "Swimming-Technique"
        elif any(term in text_lower for term in ['training', 'workout', 'set']):
            return "Swimming-Training"
        elif any(term in text_lower for term in ['rule', 'regulation', 'competition']):
            return "Swimming-Rules"
        elif any(term in text_lower for term in ['safety', 'rescue', 'lifeguard']):
            return "Swimming-Safety"
        else:
            return "Swimming-General"
    
    def identify_stroke(self, text: str) -> str:
        """Identifier la nage concernée"""
        text_lower = text.lower()
        
        if any(term in text_lower for term in ['freestyle', 'front crawl', 'crawl']):
            return "Freestyle"
        elif any(term in text_lower for term in ['backstroke', 'back stroke']):
            return "Backstroke"
        elif any(term in text_lower for term in ['breaststroke', 'breast stroke']):
            return "Breaststroke"
        elif any(term in text_lower for term in ['butterfly', 'fly stroke']):
            return "Butterfly"
        elif any(term in text_lower for term in ['individual medley', 'im', 'medley']):
            return "Medley"
        else:
            return "All-Strokes"
    
    def identify_level(self, text: str) -> str:
        """Identifier le niveau"""
        text_lower = text.lower()
        
        if any(term in text_lower for term in ['beginner', 'novice', 'learning', 'basic']):
            return "Beginner"
        elif any(term in text_lower for term in ['intermediate', 'improving']):
            return "Intermediate"
        elif any(term in text_lower for term in ['advanced', 'competitive', 'elite', 'expert']):
            return "Advanced"
        else:
            return "All-Levels"
    
    def create_swimming_embeddings(self, save_path: str = "output/embeddings_swimming.pkl"):
        """Créer embeddings optimisés pour natation"""
        print("🏊‍♂️ Création des embeddings spécialisés natation...")
        
        # Préparer textes optimisés natation
        texts = []
        for chunk in tqdm(self.chunks, desc="Optimisation textes natation"):
            swimming_text = self.optimize_swimming_text(chunk)
            texts.append(swimming_text)
        
        # Créer embeddings
        print("🔄 Génération embeddings natation...")
        self.embeddings = self.model.encode(
            texts,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        
        # Sauvegarder
        with open(save_path, 'wb') as f:
            pickle.dump({
                'embeddings': self.embeddings,
                'chunks': self.chunks
            }, f)
        
        print(f"✅ Embeddings natation sauvegardés: {save_path}")
        print(f"📊 Shape: {self.embeddings.shape}")
        
        self.create_faiss_index()
    
    def create_faiss_index(self):
        """Créer index FAISS optimisé"""
        print("🔄 Création index natation...")
        dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(self.embeddings.astype('float32'))
        print("✅ Index natation créé !")
    
    def expand_swimming_query(self, query: str) -> str:
        """Enrichir la requête avec vocabulaire natation"""
        query_lower = query.lower()
        expanded_terms = []
        
        # Expansion avec synonymes natation
        for key, synonyms in self.swimming_terms.items():
            if any(syn in query_lower for syn in synonyms):
                expanded_terms.extend(synonyms)
        
        # Requête enrichie
        if expanded_terms:
            expanded_query = query + " " + " ".join(set(expanded_terms))
        else:
            expanded_query = query
        
        return expanded_query
    
    def swimming_search(self, query: str, top_k: int = 5, min_score: float = 0.15) -> List[Dict]:
        """
        Recherche optimisée pour questions natation
        """
        # Enrichir la requête
        expanded_query = self.expand_swimming_query(query)
        
        # Encoder
        query_embedding = self.model.encode([expanded_query], normalize_embeddings=True)
        
        # Rechercher
        search_k = min(top_k * 2, len(self.chunks))
        scores, indices = self.index.search(query_embedding.astype('float32'), search_k)
        
        # Filtrer et ranker
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if score < min_score:
                continue
            
            chunk = self.chunks[idx].copy()
            chunk['similarity_score'] = float(score)
            chunk['swimming_type'] = self.identify_swimming_type(chunk['text'])
            chunk['stroke'] = self.identify_stroke(chunk['text'])
            chunk['level'] = self.identify_level(chunk['text'])
            chunk['rank'] = len(results) + 1
            
            results.append(chunk)
            
            if len(results) >= top_k:
                break
        
        return results
    
    def format_swimming_response(self, results: List[Dict], query: str) -> Dict:
        """Formater réponse spécialisée natation"""
        if not results:
            return {
                'query': query,
                'text_response': f"No swimming content found for: '{query}'",
                'images': [],
                'sources': [],
                'confidence': 0.0
            }
        
        response = {
            'query': query,
            'text_response': "",
            'images': [],
            'sources': [],
            'confidence': results[0]['similarity_score'],
            'swimming_topics_found': []
        }
        
        # Identifier les sujets trouvés
        topics = set()
        for result in results:
            topics.add(f"{result['stroke']} - {result['swimming_type']} - {result['level']}")
        
        response['swimming_topics_found'] = list(topics)
        
        # Construire réponse
        texts = []
        for i, result in enumerate(results):
            source = f"{result['pdf'].replace('.pdf', '')} (p.{result['page']})"
            topic = f"{result['stroke']} {result['swimming_type']} ({result['level']})"
            
            formatted = f"**{i+1}. {topic}** - Score: {result['similarity_score']:.3f}\n"
            formatted += f"📖 Source: {source}\n"
            formatted += f"📝 {result['text']}\n"
            
            texts.append(formatted)
            
            # Images avec contexte natation
            for img in result['images']:
                if img['path'] not in [i['path'] for i in response['images']]:
                    img['source'] = source
                    img['swimming_context'] = topic
                    response['images'].append(img)
            
            if source not in response['sources']:
                response['sources'].append(source)
        
        response['text_response'] = "\n".join(texts)
        return response

# UTILISATION OPTIMISÉE NATATION
def main():
    # RAG spécialisé natation
    rag = SwimmingOptimizedRAG()
    
    # Charger chunks natation
    rag.load_swimming_chunks("output/chunks_prepared.json")
    
    # Créer embeddings spécialisés
    rag.create_swimming_embeddings()
    
    # Tests avec questions natation spécifiques
    swimming_queries = [
        "freestyle technique breathing",
        "backstroke drills for beginners",
        "butterfly stroke mechanics",
        "breaststroke kick timing",
        "swimming workout sets",
        "competitive swimming training",
        "how to improve stroke efficiency",
        "swimming safety techniques",
        "individual medley training",
        "sprint swimming drills"
    ]
    
    print("\n🏊‍♂️ TESTS RECHERCHE NATATION OPTIMISÉE:")
    for query in swimming_queries:
        print(f"\n❓ '{query}'")
        results = rag.swimming_search(query, top_k=3)
        
        if results:
            for result in results:
                print(f"  ✅ [{result['stroke']}] [{result['swimming_type']}] [{result['level']}]")
                print(f"     Score: {result['similarity_score']:.3f}")
                print(f"     📖 {result['pdf']} - Page {result['page']}")
                print(f"     🖼️  Images: {len(result['images'])}")
                print(f"     📝 {result['text'][:100]}...")
        else:
            print("  ❌ No relevant swimming content found")

if __name__ == "__main__":
    main()

🏊‍♂️ Chargement du modèle optimisé pour natation...


c:\ProgramData\miniconda3\envs\rag-env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


✅ Modèle spécialisé natation chargé !
🏊‍♂️ 31021 chunks de natation chargés

🏊‍♂️ ANALYSE CONTENU NATATION:
📚 Techniques: 7392 chunks
🏃‍♂️ Exercices: 6227 chunks
👶 Débutant: 890 chunks
🏆 Avancé: 1610 chunks
🏊‍♀️ Freestyle: 2478 chunks
🏊‍♂️ Backstroke: 5249 chunks
🐸 Breaststroke: 1789 chunks
🦋 Butterfly: 1660 chunks
🖼️  Avec images: 7033 chunks
🏊‍♂️ Création des embeddings spécialisés natation...


Optimisation textes natation: 100%|██████████| 31021/31021 [00:02<00:00, 10475.15it/s]


🔄 Génération embeddings natation...


Batches: 100%|██████████| 970/970 [2:51:21<00:00, 10.60s/it]  


✅ Embeddings natation sauvegardés: output/embeddings_swimming.pkl
📊 Shape: (31021, 768)
🔄 Création index natation...
✅ Index natation créé !

🏊‍♂️ TESTS RECHERCHE NATATION OPTIMISÉE:

❓ 'freestyle technique breathing'
  ✅ [Freestyle] [Swimming-Technique] [All-Levels]
     Score: 0.826
     📖 SWIMMING.pdf - Page 45
     🖼️  Images: 0
     📝 Breathing and Coordination. Coordinate
the arm movements and scissors kick
as in the sidestroke. If ...
  ✅ [Freestyle] [Swimming-Technique] [All-Levels]
     Score: 0.814
     📖 SWIMMING FOR TOTAL FITNESS.pdf - Page 80
     🖼️  Images: 0
     📝 By combining your breathing skills with one other skill at a time, you
are coordinating in gradual s...
  ✅ [Freestyle] [Swimming-Technique] [Beginner]
     Score: 0.807
     📖 SWIMMING FOR TOTAL FITNESS.pdf - Page 74
     🖼️  Images: 0
     📝 L
ESSON 4
T
his is the breakthrough lesson for new swimmers. You’re now ready to
take the final step...

❓ 'backstroke drills for beginners'
  ✅ [Backstroke] [Swimming-